# 302 · Production Deployment: BigQuery

While DuckDB is excellent for development and local prototyping, production A/B test orchestration often requires a centralized, scalable data warehouse. 

Because `EarlySign` uses **Ibis**, transitioning from local files to **Google BigQuery** is seamless. The code logic—including controllers, projectors, and analysis—remains exactly the same.

In this tutorial, we will:
- Connect to **BigQuery Sandbox**.
- Initialize a production-scale `Ledger`.
- Learn how `EarlySign` automatically optimizes BigQuery performance via clustering.

## 1. Connection Setup

You will need a Google Cloud Project with the BigQuery API enabled. BigQuery Sandbox is free and does not require a credit card.

In [1]:
# 1. Define your projects
# Execution Project: Handles query compute and billing
BILLING_PROJECT_ID = "your-execution-project-id"

# Storage Project: Holds the BigQuery dataset and tables
STORAGE_PROJECT_ID = "your-storage-project-id"
DATASET_ID = "earlysign_production"
TABLE_NAME = "experiment_ledger"

# 2. Connect via Ibis
# We specify the dataset location as 'project.dataset' to separate storage from billing.
# con = ibis.bigquery.connect(
#     project_id=BILLING_PROJECT_ID,
#     dataset_id=f"{STORAGE_PROJECT_ID}.{DATASET_ID}"
# )
#
# Alternatively, use the connection URL:
# con = ibis.connect(f"bigquery://{BILLING_PROJECT_ID}/{STORAGE_PROJECT_ID}.{DATASET_ID}")

print(f"Target Backend: BigQuery ({STORAGE_PROJECT_ID}.{DATASET_ID}.{TABLE_NAME})")

Target Backend: BigQuery (your-storage-project-id.earlysign_production.experiment_ledger)


## 2. Infrastructure Management and Clustering

One of the key advantages of `EarlySign` is that it can manage its own infrastructure. `ledger.ensure()` will create the BigQuery table with the correct schema if it doesn't already exist.

**Automatic Optimization**: 
To ensure high-performance querying in large event logs, `EarlySign` uses **BigQuery Clustering**. When you initialize a `Ledger` with a `ledger_id`, `ensure()` automatically configures the BigQuery table to cluster by the `ledger_id` column. This physically organizes rows belonging to the same experiment together, drastically reducing data scan costs and latency for analysis.

In [2]:
# # By providing a ledger_id, we enable BigQuery clustering for this table.
# ledger = Ledger(con, TABLE_NAME, ledger_id="prod_experiment_001")
#
# # ensure() maps types to BigQuery native types and sets up the clustering columns.
# ledger.ensure()
print("Ledger infrastructure verified in BigQuery (with clustering on ledger_id).")

Ledger infrastructure verified in BigQuery (with clustering on ledger_id).


## 3. High-Concurrency Patterns

In a production environment, multiple processes might write to the ledger simultaneously. 
- **Immutable Append**: Since we only *append* events (no updates), there are no row-level locks.
- **Distributed Analytics**: Your analytical worker can run in a separate Cloud Function or Kubernetes Pod, reading the latest state from BigQuery on-demand. Clustering ensures these reads are highly efficient.

In [3]:
# trial = JennisonTurnbull2000Controller(ledger)
# Everything else is the same!
# trial.update(batch)
# trial.report_progress()

## 4. Conclusion

Congratulations! You have completed the `EarlySign` tutorial series. You now know how to:
1. Manage data with **Event Sourcing** in the Ledger.
2. Build complex analysis with **Sessions and Projectors**.
3. Apply **GSD, SSR, and AVI** to your experiments.
4. Validate strategies via **Backtesting**.
5. Scale to **Production** with BigQuery optimizations like clustering.

For more technical details, check out the [Explanation](../explanation/README.rst) section, the [ADR on Clustering](../reference/ADR/ADR_008.rst), or the [API Reference](../reference/README.rst).